# Balanced Trees, Hashing, and Tries

The binary search tree from Part III has a beautiful $O(\log n)$ promise and one fatal flaw: insert sorted data and it collapses into a linked list. This lab starts by *measuring* that collapse, then fixes it two ways (AVL rotations and red-black colours), and then builds the structures that beat trees at their own game — hash tables, tries, and skip lists.

Every structure here has an **invariant**: a property that must hold after every operation. The house rule of this lab is that no invariant is asserted in prose without a runnable checker that verifies it. `is_bst`, `is_avl`, `rb_validate` and the tombstone probe are all in that spirit — write the checker first, and the implementation stops lying to you.

**How to use this notebook:** run cells top to bottom; later sections reuse earlier classes. Companion reading: Chapters 35 and 36.

## 1. The wound, measured

Chapter 20's insert walks down from the root and plants the new key where the search falls off the tree. Nothing in it looks at *shape*. So the shape is decided entirely by arrival order, and there is one arrival order that is a disaster: sorted.

The **height** of a tree is the number of edges on its longest root-to-leaf path, and it is exactly the worst-case number of comparisons a search can cost.

In [ ]:
import random

class Node:
    def __init__(self, key):
        self.key = key
        self.left = self.right = None

def insert(root, key):
    """Chapter 20's insert, written iteratively.

    The recursive version is prettier, but a sorted insert of 1000 keys builds
    a 1000-deep chain and every recursive walk down it would blow Python's
    stack -- which is itself a symptom of the problem this section is about.
    """
    if root is None:
        return Node(key)
    node = root
    while True:
        if key < node.key:
            if node.left is None:
                node.left = Node(key)
                return root
            node = node.left
        elif key > node.key:
            if node.right is None:
                node.right = Node(key)
                return root
            node = node.right
        else:
            return root                    # duplicate: ignore

def height(node):
    """Longest root-to-leaf path in edges, measured level by level."""
    if node is None:
        return -1                          # empty tree -1, a single leaf 0
    h, level = -1, [node]
    while level:
        h += 1
        level = [c for n in level for c in (n.left, n.right) if c is not None]
    return h

def build(order):
    root = None
    for k in order:
        root = insert(root, k)
    return root

def probe(root, key):
    """Search, counting comparisons - the number that actually costs you."""
    steps = 0
    while root is not None:
        steps += 1
        if key == root.key:
            return steps
        root = root.left if key < root.key else root.right
    return steps

import math
n = 1023
random.seed(35)
orders = {
    "sorted 1..1023": list(range(1, n + 1)),
    "reverse sorted": list(range(n, 0, -1)),
    "shuffled": random.sample(range(1, n + 1), n),
}
print(f"{n} keys, ideal height = {math.floor(math.log2(n))}\n")
print(f"{'arrival order':<18}{'height':>8}{'worst search':>14}{'search for 1023':>17}")
for label, order in orders.items():
    tree = build(order)
    print(f"{label:<18}{height(tree):>8}{height(tree) + 1:>14}"
          f"{probe(tree, n):>17}")

A thousand keys arriving in sorted order produce a tree of height 1022 — a linked list wearing a tree's clothes. Every `.left` pointer is `None`, every search walks the whole chain, and the $O(\log n)$ promise is gone. Shuffled input behaves close to the ideal, which is the trap: your tests use shuffled data and your production system imports a sorted CSV.

Sorted input is not exotic. Database keys arrive in insertion order, timestamps arrive in time order, and "import the file we exported last week" arrives sorted. Any structure whose performance depends on arrival order will eventually meet the arrival order that breaks it.

## 2. Rotations: changing shape without changing order

The repair tool is a **rotation**: a local pointer rearrangement that changes the tree's *shape* while preserving its in-order sequence. It touches three pointers and is therefore $O(1)$, no matter how big the tree is.

A right rotation at `y` promotes `y`'s left child `x` to `y`'s position. `x`'s right subtree — the keys between `x` and `y` — becomes `y`'s new left subtree, which is exactly where those keys still belong.

```text
      y                 x
     / \               / \
    x   C    ---->    A   y
   / \                   / \
  A   B                 B   C
```

Read both pictures in order: `A x B y C` before, `A x B y C` after. That equality *is* the correctness proof, and the cell below turns it into an assertion.

In [ ]:
def rotate_right(y):
    x = y.left
    y.left = x.right          # B moves across
    x.right = y
    return x                  # x is the new subtree root

def rotate_left(x):
    y = x.right
    x.right = y.left
    y.left = x
    return y

def in_order(node, out=None):
    if out is None:
        out = []
    if node is not None:
        in_order(node.left, out)
        out.append(node.key)
        in_order(node.right, out)
    return out

def is_bst(node, lo=None, hi=None):
    """Every key must lie strictly inside the window inherited from ancestors."""
    if node is None:
        return True
    if lo is not None and node.key <= lo:
        return False
    if hi is not None and node.key >= hi:
        return False
    return (is_bst(node.left, lo, node.key)
            and is_bst(node.right, node.key, hi))

def show(node, indent=""):
    if node is None:
        return
    show(node.right, indent + "     ")
    print(f"{indent}{node.key}")
    show(node.left, indent + "     ")

root = build([30, 20, 40, 10, 25, 5])      # leaning left
before = in_order(root)
print("before rotation, height", height(root), " in-order", before)
show(root)

root = rotate_right(root)
print("\nafter rotate_right at the root, height", height(root),
      " in-order", in_order(root))
show(root)

assert in_order(root) == before, "a rotation must never reorder keys"
assert is_bst(root)
print("\nin-order sequence unchanged and still a valid BST:", in_order(root) == before)

# And a rotation is its own inverse in the other direction:
root = rotate_left(root)
print("rotate_left undoes it:", in_order(root) == before, " height", height(root))

The tree leaned left, one right rotation moved the root and dropped the height from 3 to 2, and the in-order sequence came through untouched. That is the whole guarantee a balancer needs: it may reshape freely, and the search property survives automatically. (Rotating back the other way restores the original shape — a rotation is exactly reversible, which is why `rotate_left` is not a separate idea but the same idea mirrored.)

One rotation is not always enough. If the tree leans left but the *child* leans right (a "zig-zag"), a single right rotation just moves the problem to the other side. The fix is a **double rotation**: rotate the child left first to straighten the zig-zag into a straight line, then rotate the parent right.

### Micro-exercise: the double rotation

Write `rotate_left_right(z)` for the case where `z`'s left child leans right: rotate `z.left` to the left, then rotate `z` to the right. Verify the in-order sequence is preserved and that the height drops.

In [ ]:
def rotate_left_right(z):
    # your code here: z.left = rotate_left(z.left), then return rotate_right(z)
    return z

# Uncomment to test:
# zig = build([30, 10, 20])          # leans left, but its child leans right
# print("before:", in_order(zig), "height", height(zig))
# fixed = rotate_left_right(zig)
# print("after :", in_order(fixed), "height", height(fixed))
# assert in_order(fixed) == [10, 20, 30] and height(fixed) == 1

## 3. AVL trees: rebalance on the way back up

An **AVL tree** (Adelson-Velsky and Landis, 1962) adds one invariant to the BST:

> For every node, the heights of its two subtrees differ by at most 1.

That number, $h(\text{left}) - h(\text{right})$, is the node's **balance factor**, and the invariant is $|bf| \le 1$. It is strong enough to force height $O(\log n)$ and weak enough to be maintained with $O(1)$ work per level.

Recomputing heights on demand would cost $O(n)$ per node, so each node **caches** its height. Insertion is Chapter 20's recursion with one change: `return self._rebalance(node)` instead of `return node`. On the way back up, each node refreshes its cached height, looks at its balance factor, and applies one of four repairs — LL, RR, LR, RL — named for the direction of the child and grandchild that caused the trouble.

In [ ]:
class AVLNode:
    __slots__ = ("key", "left", "right", "height")
    def __init__(self, key):
        self.key = key
        self.left = self.right = None
        self.height = 0

def h(node):
    return -1 if node is None else node.height

class AVLTree:
    def __init__(self):
        self.root = None
        self.rotations = 0

    # ---- the two primitives, now maintaining cached heights
    def _rotate_right(self, y):
        x = y.left
        y.left = x.right
        x.right = y
        y.height = 1 + max(h(y.left), h(y.right))     # y first: it is now lower
        x.height = 1 + max(h(x.left), h(x.right))
        self.rotations += 1
        return x

    def _rotate_left(self, x):
        y = x.right
        x.right = y.left
        y.left = x
        x.height = 1 + max(h(x.left), h(x.right))
        y.height = 1 + max(h(y.left), h(y.right))
        self.rotations += 1
        return y

    def _bf(self, node):
        return 0 if node is None else h(node.left) - h(node.right)

    def _rebalance(self, node):
        node.height = 1 + max(h(node.left), h(node.right))
        bf = self._bf(node)
        if bf > 1:                                    # left-heavy
            if self._bf(node.left) < 0:               # ...and the child leans right: LR
                node.left = self._rotate_left(node.left)
            return self._rotate_right(node)           # LL (or the second half of LR)
        if bf < -1:                                   # right-heavy
            if self._bf(node.right) > 0:              # RL
                node.right = self._rotate_right(node.right)
            return self._rotate_left(node)            # RR (or the second half of RL)
        return node

    def insert(self, key):
        self.root = self._insert(self.root, key)

    def _insert(self, node, key):
        if node is None:
            return AVLNode(key)
        if key < node.key:
            node.left = self._insert(node.left, key)
        elif key > node.key:
            node.right = self._insert(node.right, key)
        else:
            return node                               # duplicate: ignore
        return self._rebalance(node)

    def height(self):
        return h(self.root)

    def in_order(self):
        out, stack, node = [], [], self.root
        while stack or node:
            while node:
                stack.append(node)
                node = node.left
            node = stack.pop()
            out.append(node.key)
            node = node.right
        return out

def is_avl(root):
    """Verify BOTH invariants: BST ordering and |balance factor| <= 1."""
    def check(node):
        if node is None:
            return -1, True                # (height, ok)
        lh, lok = check(node.left)
        rh, rok = check(node.right)
        ok = lok and rok and abs(lh - rh) <= 1
        if node.left is not None and node.left.key >= node.key:
            ok = False
        if node.right is not None and node.right.key <= node.key:
            ok = False
        if node.height != 1 + max(lh, rh):             # the cache must be honest
            ok = False
        return 1 + max(lh, rh), ok
    return check(root)[1]

print("AVLTree defined; is_avl checks ordering, balance, and the cached heights.")

Now the head-to-head that motivates the whole chapter. Insert the keys 1 through 31 — the single worst possible order — into both structures.

In [ ]:
def plain_bst_height(keys):
    root = None
    for k in keys:
        root = insert(root, k)
    return height(root)

keys = list(range(1, 32))              # 1, 2, 3, ..., 31 -- the worst case
tree = AVLTree()
for k in keys:
    tree.insert(k)

print(f"{'structure':<24}{'height':>8}{'worst-case search':>19}")
print(f"{'plain BST (Chapter 20)':<24}{plain_bst_height(keys):>8}"
      f"{plain_bst_height(keys) + 1:>19}")
print(f"{'AVL tree':<24}{tree.height():>8}{tree.height() + 1:>19}")
print(f"\nrotations performed: {tree.rotations}")
print("in-order traversal still sorted:", tree.in_order() == keys)
print("invariants hold:", is_avl(tree.root))

print(f"\n{'n':>7}{'AVL height':>12}{'ideal':>8}{'guaranteed ceiling':>21}"
      f"{'rotations':>11}")
for n in (31, 255, 1023, 4095):
    t = AVLTree()
    for k in range(1, n + 1):
        t.insert(k)
    assert is_avl(t.root)
    print(f"{n:>7}{t.height():>12}{math.floor(math.log2(n)):>8}"
          f"{1.44 * math.log2(n):>21.2f}{t.rotations:>11}")

Thirty-one keys in the worst possible order, and the AVL tree is a *perfect* tree — the theoretical minimum, since 31 nodes need 5 levels. The plain BST is a 31-node chain. A couple of dozen $O(1)$ rotations bought a sixfold reduction in worst-case search, and the in-order traversal came through untouched exactly as the rotation guarantee promised.

The second table is the theoretical bound made concrete. An AVL tree's height is at most about $1.44 \log_2 n$ — never worse than roughly 44% more comparisons than a perfectly balanced tree, *guaranteed*, for any input whatsoever. And look at what actually happened: for sequential inserts the measured height equals the ideal $\lfloor \log_2 n \rfloor$ at every size, so the tree is not merely balanced but *perfect*, well under the ceiling. Note the rotation count too — it grows roughly linearly in $n$, which is to say roughly one rotation per couple of insertions, each of them $O(1)$.

Deletion is where AVL earns its reputation for being fiddly. It reuses Chapter 20's three cases (leaf, one child, two children via the in-order successor) and then rebalances every node on the path back up — and unlike insertion, where one rotation always suffices, a single deletion can require $O(\log n)$ rotations, because fixing one node's balance can shorten its subtree and unbalance its parent.

## 4. Red-black trees: the same job, done with colours

AVL keeps the tree *very* balanced and pays for it with rotations. A **red-black tree** relaxes the balance target and does noticeably less rearranging — which is why it, not AVL, is what `std::map`, Java's `TreeMap`, and most kernel schedulers use.

It maintains five properties:

1. Every node is either **red** or **black**.
2. The root is black.
3. Every leaf (the `None` children) counts as black.
4. **A red node has no red child** (equivalently: no two reds in a row on any path).
5. Every path from a given node down to any leaf contains the **same number of black nodes** — that count is the node's **black height**.

Properties 4 and 5 together are what bound the height. Property 5 says the "black skeleton" is perfectly balanced; property 4 says reds can at most double a path's length. Hence height $\le 2\log_2(n+1)$ — a weaker bound than AVL's $1.44 \log_2 n$, bought with fewer rotations.

Five properties is four more than AVL's one, which makes a validator that *names the broken property* the single most useful piece of code you can write here.

In [ ]:
RED, BLACK = "R", "B"

class RBNode:
    __slots__ = ("key", "color", "left", "right")
    def __init__(self, key, color=RED, left=None, right=None):
        self.key, self.color, self.left, self.right = key, color, left, right

def rb_validate(root):
    """Return a list of violated properties, by number. Empty list = valid."""
    problems = []
    if root is not None and root.color != BLACK:
        problems.append("property 2: the root is not black")

    def walk(node):
        """Returns the black height, or None if a subtree is already broken."""
        if node is None:
            return 1                       # a None leaf counts as one black
        if node.color not in (RED, BLACK):
            problems.append(f"property 1: {node.key} has colour {node.color!r}")
        if node.color == RED:
            for child in (node.left, node.right):
                if child is not None and child.color == RED:
                    problems.append(f"property 4: red {node.key} has red child "
                                    f"{child.key}")
        lh, rh = walk(node.left), walk(node.right)
        if lh is not None and rh is not None and lh != rh:
            problems.append(f"property 5: black heights differ under {node.key} "
                            f"({lh} on the left, {rh} on the right)")
            return None
        black_here = 1 if node.color == BLACK else 0
        return None if lh is None or rh is None else lh + black_here

    def bst_check(node, lo=None, hi=None):
        if node is None:
            return
        if (lo is not None and node.key <= lo) or (hi is not None and node.key >= hi):
            problems.append(f"BST order: {node.key} is in the wrong subtree")
        bst_check(node.left, lo, node.key)
        bst_check(node.right, node.key, hi)

    walk(root)
    bst_check(root)
    return problems

def N(key, color, left=None, right=None):
    return RBNode(key, color, left, right)

VALID = N(13, BLACK,
          N(8, RED, N(1, BLACK), N(11, BLACK)),
          N(17, RED, N(15, BLACK), N(25, BLACK)))

RED_ROOT = N(13, RED, N(8, BLACK), N(17, BLACK))
RED_RED = N(13, BLACK, N(8, RED, N(5, RED), None), N(17, BLACK))
LOPSIDED = N(13, BLACK, N(8, BLACK, N(1, BLACK), None), N(17, BLACK))

for label, tree in [("textbook-valid tree", VALID), ("red root", RED_ROOT),
                    ("red node with a red child", RED_RED),
                    ("unequal black heights", LOPSIDED)]:
    issues = rb_validate(tree)
    print(f"{label:<28} {'VALID' if not issues else issues[0]}")

Four trees, three diagnoses, each naming the property it broke. Note what `LOPSIDED` looks like: it has no colour violation at all — every node is black, so no two reds can be adjacent — and it is still not a red-black tree. The validator points straight at node 8: black height 2 down the left (through node 1) against 1 down the missing right child. Property 5 is the one people forget, and it is the one doing the actual balancing work.

Insertion colours the new node **red** first, then repairs. Red is the optimistic choice: it cannot break property 5, only property 4, and a property-4 violation is local. If the new node's *uncle* is also red, a recolour fixes it and the problem may move two levels up; if the uncle is black, one or two rotations fix it permanently. That is why red-black insertion needs at most two rotations regardless of tree size, where AVL may rotate at every level on the way up.

| | AVL | red-black |
|---|---|---|
| Invariant | $\lvert h_L - h_R \rvert \le 1$ everywhere | the five colour properties |
| Height bound | $\approx 1.44 \log_2 n$ | $\le 2 \log_2 (n+1)$ |
| Rotations per insert | up to $O(\log n)$ | at most 2 |
| Rotations per delete | up to $O(\log n)$ | at most 3 |
| Better for | lookup-heavy workloads | update-heavy workloads |
| Found in | in-memory indexes, some databases | `std::map`, `TreeMap`, Linux CFS |

## 5. Hash tables: turning a key into an address

A tree gets you to a key in $O(\log n)$ comparisons. A **hash table** aims to get there in $O(1)$ *no comparisons at all*: run the key through a hash function, take the result modulo the table size, and that is the bucket.

Everything then depends on one thing — whether the hash function spreads keys evenly. Below are three: a naive character sum, Bernstein's classic `djb2`, and Python's built-in `hash`. All the keys are distinct five-letter words, so a perfect hash would produce as many distinct outputs as words.

In [ ]:
WORDS = """
about above abuse actor adapt admit adopt after again agent agree ahead alarm
album alert alien align alike alive allow alone along alter among angel anger
angle ankle apart apple apply arena argue arise armor array arrow aside asset
audio audit avoid awake award aware badly baker bases basic beach began begin
begun being below bench birth black blade blame blank blast blend blind block
blood board boost booth bound brain brand brave bread break breed brick brief
bring broad broke brown build built burst buyer cable catch cause chain chair
chaos charm chart chase cheap check chest chief child chose civil claim class
clean clear click climb clock close cloud coach coast could count court cover
craft crash cream crime cross crowd crown curve cycle daily dance dated dealt
death debut delay depth doing doubt dozen draft drama drawn dream dress drink
drive drove dying eager early earth eight elite empty enemy enjoy enter entry
equal error event every exact exist extra
notes tones onset steno seton
cried dicer riced cider
spare pears reaps parse spear
""".split()

def weak_hash(s):
    """Sum the character codes. Simple, fast -- and badly broken."""
    return sum(ord(c) for c in s)

def djb2(s):
    """Bernstein's classic: h = h*33 + c, kept to 32 bits."""
    h = 5381
    for c in s:
        h = (h * 33 + ord(c)) & 0xFFFFFFFF
    return h

print(f"{len(WORDS)} words, all five letters, all distinct\n")
print(f"{'hash':<14}{'distinct outputs':>18}{'smallest':>12}{'largest':>12}{'span':>12}")
w = [weak_hash(s) for s in WORDS]
d = [djb2(s) for s in WORDS]
print(f"{'weak_hash':<14}{len(set(w)):>18}{min(w):>12}{max(w):>12}{max(w) - min(w):>12}")
print(f"{'djb2':<14}{len(set(d)):>18}{min(d):>12}{max(d):>12}{max(d) - min(d):>12}")
print(f"{'built-in hash':<14}{len(set(hash(s) for s in WORDS)):>18}"
      f"{'(varies)':>12}{'(varies)':>12}{'(varies)':>12}")

print("\nanagram families collapse under weak_hash, because addition forgets order:")
for family in (["notes", "tones", "onset", "steno", "seton"],
               ["spare", "pears", "reaps", "parse", "spear"]):
    print(f"  {family} -> weak_hash all = {weak_hash(family[0])}, "
          f"djb2 distinct = {len({djb2(x) for x in family})}")

Read the first row again. Nearly two hundred distinct words, and `weak_hash` produces only 54 distinct outputs across a span of 63 — because summing character codes discards *order*, so every anagram collides, and five lower-case letters can only sum to a narrow range. Every collision is a bucket you have to search linearly.

`djb2` gives one output per word across a span of tens of millions. The multiply-by-33 is what makes position matter: each character is shifted before the next is added, so `notes` and `onset` land nowhere near each other.

Buckets are what actually matter, though, so look at where the keys land. **Chi-squared** measures how far a distribution is from uniform: for $m$ buckets the expected value under a perfect hash is about $m$, and bigger means lumpier.

In [ ]:
import matplotlib.pyplot as plt

M = 128

def bucket_counts(fn, words, m=M):
    counts = [0] * m
    for word in words:
        counts[fn(word) % m] += 1
    return counts

def chi_squared(counts, n):
    expected = n / len(counts)
    return sum((c - expected) ** 2 / expected for c in counts)

n_words = len(WORDS)
print(f"{n_words} words into {M} buckets  "
      f"(a uniform hash gives chi2 near {M}, "
      f"about {M * math.exp(-n_words / M):.0f} empty buckets)\n")
alpha = n_words / M
print(f"load factor alpha = {alpha:.2f}, so a good hash should need about "
      f"1 + alpha/2 = {1 + alpha / 2:.2f} comparisons\n")
print(f"{'hash':<14}{'chi2':>9}{'empty':>8}{'fullest':>9}{'mean probe if chained':>24}")

fns = [("weak_hash", weak_hash), ("djb2", djb2), ("built-in hash", hash)]
all_counts = []
for name, fn in fns:
    counts = bucket_counts(fn, WORDS)
    all_counts.append(counts)
    # expected comparisons for a successful search: average over keys of
    # (position in its chain), approximated by (chain length + 1) / 2
    mean_probe = sum(c * (c + 1) / 2 for c in counts) / n_words
    print(f"{name:<14}{chi_squared(counts, n_words):>9.0f}{counts.count(0):>8}"
          f"{max(counts):>9}{mean_probe:>24.2f}")

fig, axes = plt.subplots(3, 1, figsize=(8, 6), sharex=True, sharey=True)
for ax, (name, _), counts in zip(axes, fns, all_counts):
    ax.bar(range(M), counts, width=1.0)
    ax.set_ylabel("keys")
    ax.set_title(f"{name}: {counts.count(0)} of {M} buckets empty, "
                 f"fullest holds {max(counts)}", fontsize=10)
axes[-1].set_xlabel("bucket index")
fig.tight_layout()

The three panels are the same keys, the same table, three hash functions, and the middle column of the printed table is the damning one. `weak_hash` cannot even reach most of the table: its whole output span is 63, so with 128 buckets **more than half of them are unreachable by construction** — 74 sit empty no matter how many words you add, while the buckets it can reach hold up to 11 keys each. Its chi-squared is several times the uniform baseline and its mean probe count is well above what the load factor predicts.

`djb2` and Python's built-in spread the keys across the whole table, and their mean probe counts land essentially on $1 + \alpha/2$ — the formula for chained hashing, confirmed rather than asserted. That is the number to remember: search cost depends on the **load factor**, not on the table's absolute size.

Two practical notes. Python's `hash` is **randomly salted per process** for strings, as a defence against attackers who deliberately craft colliding keys — so its bucket layout changes between runs while `djb2`'s does not. And table size matters: with a power-of-two size, `% m` keeps only the low bits, so a hash whose entropy sits in its high bits will look terrible; a prime size mixes all the bits in. Good hash functions plus power-of-two sizes are the common modern combination, because the masking is one instruction.

## 6. The tombstone bug

Chaining is not the only collision strategy. **Open addressing** keeps everything in one array: if a key's slot is taken, probe the next one until you find a free slot. Lookups follow the same probe sequence and — crucially — **stop at the first empty slot**, because an empty slot proves the key was never inserted.

Now delete a key by emptying its slot, and you have punched a hole in the middle of somebody else's probe sequence. This bug has shipped in real code more than once.

In [ ]:
M2 = 8
slots = [None] * M2

def raw_insert(k):
    i = k % M2
    while slots[i] is not None:
        i = (i + 1) % M2
    slots[i] = k
    return i

def raw_find(k):
    """Stop at the first empty slot - the standard, and here fatal, rule."""
    i = k % M2
    while slots[i] is not None:
        if slots[i] == k:
            return i
        i = (i + 1) % M2
    return None

for k in (1, 9, 17):                    # 1 % 8 == 9 % 8 == 17 % 8 == 1
    print(f"insert {k:>2} -> slot {raw_insert(k)}")
print("table:", slots)
print("find(17) ->", raw_find(17), " (correct: 17 lives in slot 3)")

slots[2] = None                         # "delete" 9 by clearing its slot
print("\nafter clearing slot 2 (deleting 9):", slots)
print("find(17) ->", raw_find(17), " <-- WRONG: reported missing")
print("...but 17 is still sitting in slot", slots.index(17))

Deleting 9 deleted 17 as well, as far as anyone can tell. The search for 17 starts at slot 1, steps to slot 2, finds it empty, and concludes — correctly, by its own rule — that 17 was never inserted. The value is intact, occupying memory, and permanently unreachable. No exception, no warning; just a table that has started lying. Delete a few thousand keys from a busy cache and you lose entries you never touched.

The fix has a name: a **tombstone**. Deleting writes a special marker meaning *"something used to be here — keep probing"*. Searches walk past tombstones; insertions may reuse them.

In [ ]:
EMPTY = None
TOMB = object()                         # a unique marker: "deleted, keep going"

class LinearProbeMap:
    """Open addressing with linear probing, tombstones, and resizing."""

    def __init__(self, capacity=8, max_load=0.5):
        self._keys = [EMPTY] * capacity
        self._values = [None] * capacity
        self._size = 0                  # live entries
        self._used = 0                  # live entries + tombstones
        self.max_load = max_load

    def _slot(self, key, insert=False):
        """Walk the probe sequence. Returns the index to act on."""
        m = len(self._keys)
        i = hash(key) % m
        first_tomb = None
        while True:
            k = self._keys[i]
            if k is EMPTY:
                return first_tomb if (insert and first_tomb is not None) else i
            if k is TOMB:
                if first_tomb is None:
                    first_tomb = i
            elif k == key:
                return i
            i = (i + 1) % m             # keep probing PAST the tombstone

    def put(self, key, value):
        i = self._slot(key, insert=True)
        if self._keys[i] not in (EMPTY, TOMB) and self._keys[i] == key:
            self._values[i] = value
            return
        if self._keys[i] is EMPTY:
            self._used += 1
        self._keys[i], self._values[i] = key, value
        self._size += 1
        if self._used > self.max_load * len(self._keys):
            self._resize(len(self._keys) * 2)

    def get(self, key, default=None):
        i = self._slot(key)
        k = self._keys[i]
        return self._values[i] if k is not EMPTY and k is not TOMB else default

    def remove(self, key):
        i = self._slot(key)
        k = self._keys[i]
        if k is EMPTY or k is TOMB:
            raise KeyError(key)
        self._keys[i], self._values[i] = TOMB, None      # <- the fix
        self._size -= 1

    def _resize(self, capacity):
        """Rehash live entries only: this is what clears out the tombstones."""
        pairs = [(k, v) for k, v in zip(self._keys, self._values)
                 if k is not EMPTY and k is not TOMB]
        self._keys = [EMPTY] * capacity
        self._values = [None] * capacity
        self._size = self._used = 0
        for k, v in pairs:
            self.put(k, v)

    def __len__(self):
        return self._size

    def layout(self):
        return ["." if k is EMPTY else ("X" if k is TOMB else str(k))
                for k in self._keys]

m = LinearProbeMap(capacity=8)
for k in (1, 9, 17, 4):
    m.put(k, f"v{k}")
print("layout:", m.layout(), " size", len(m))
m.remove(9)
print("after remove(9):", m.layout(), " size", len(m))
print("get(17) ->", m.get(17), "  <-- found, because the search walked past X")
print("get(9)  ->", m.get(9))
m.put(33, "v33")                        # 33 % 8 == 1: may reuse the tombstone
print("after put(33):", m.layout(), " size", len(m))
print("all three still reachable:", [m.get(k) for k in (1, 17, 33)])

`get(17)` now works, because the probe walked *past* the tombstone instead of stopping at it. And `put(33)` reuses the tombstone's slot rather than extending the probe sequence, which is why `_slot` remembers the first tombstone it saw but keeps walking to check the key is not already present further along. Getting that order wrong produces duplicate keys — a bug even nastier than the one we just fixed.

Tombstones have a cost: they lengthen probe sequences without holding data. A table that is hammered with inserts and deletes fills with tombstones and slows down even though `len()` stays small. That is why `_used` counts tombstones as occupied and triggers a resize, and why the rehash copies only live entries — resizing is also the garbage collector for tombstones.

## 7. Tries: the key *is* the path

Hash tables answer "is this exact key present?" beautifully, and are useless for "which keys start with `hel`?", because hashing deliberately destroys any relationship between similar keys. A **trie** (from re*trie*val, usually pronounced "try") keeps that relationship by storing the key as a *path*: one edge per character, from the root down.

Two consequences follow immediately. Lookup costs $O(L)$ in the key's length and does **not** depend on how many keys are stored — a million-word trie searches `hello` in five steps. And every prefix query is free: walk to the prefix's node, then collect everything below it.

In [ ]:
class TrieNode:
    __slots__ = ("children", "is_word", "freq")
    def __init__(self):
        self.children = {}         # character -> TrieNode
        self.is_word = False       # does a stored key END here?
        self.freq = 0

class Trie:
    def __init__(self):
        self.root = TrieNode()
        self._count = 0

    def insert(self, word, freq=1):
        node = self.root
        for ch in word:
            node = node.children.setdefault(ch, TrieNode())
        if not node.is_word:
            self._count += 1
        node.is_word = True
        node.freq = freq

    def _node_for(self, prefix):
        node = self.root
        for ch in prefix:
            if ch not in node.children:
                return None
            node = node.children[ch]
        return node

    def search(self, word):
        node = self._node_for(word)
        return node is not None and node.is_word

    def starts_with(self, prefix):
        return self._node_for(prefix) is not None

    def collect(self, prefix):
        """Every stored word beginning with `prefix`, in alphabetical order."""
        node = self._node_for(prefix)
        out = []
        if node is None:
            return out

        def dfs(n, path):
            if n.is_word:
                out.append((prefix + path, n.freq))
            for ch in sorted(n.children):        # sorted -> alphabetical output
                dfs(n.children[ch], path + ch)
        dfs(node, "")
        return out

    def node_count(self):
        total, stack = 1, [self.root]
        while stack:
            n = stack.pop()
            total += len(n.children)
            stack.extend(n.children.values())
        return total

    def __len__(self):
        return self._count

WORD_FREQ = {"hello": 90, "help": 75, "helm": 12, "helmet": 30, "held": 44,
             "he": 500, "heap": 8, "heat": 40, "hero": 25, "her": 300,
             "cat": 60, "car": 88, "care": 40, "cart": 22, "cargo": 9}
trie = Trie()
for word, f in WORD_FREQ.items():
    trie.insert(word, f)

print(f"{len(trie)} words stored in {trie.node_count()} nodes")
print("search('help')  ->", trie.search("help"))
print("search('hel')   ->", trie.search("hel"), " (a path, but not a stored word)")
print("starts_with('hel') ->", trie.starts_with("hel"))
print("\nalphabetical, prefix 'hel':", [w for w, _ in trie.collect("hel")])
print("alphabetical, prefix 'car':", [w for w, _ in trie.collect("car")])
print("prefix 'xyz':", trie.collect("xyz"))

The distinction between `search("hel")` and `starts_with("hel")` is the whole reason `is_word` exists. `hel` is a real path through the trie — it has to be, because `hello` is stored — but no key ends there. Without that flag a trie cannot tell "a stored word" from "a prefix of one", and every prefix of every key would appear to be present.

An autocomplete needs one more thing: **ranking**. Alphabetical order is nearly useless to a user; frequency order is what they expect. Since `collect` already gathers the candidates with their frequencies, ranking is a sort — and the interesting part is what to do when there are a hundred thousand candidates below the prefix.

In [ ]:
def suggest(trie, prefix, k=5):
    """Top-k completions of `prefix`, most frequent first, ties alphabetical."""
    candidates = trie.collect(prefix)
    return sorted(candidates, key=lambda wf: (-wf[1], wf[0]))[:k]

for prefix in ["he", "hel", "c", "ca", "z"]:
    ranked = suggest(trie, prefix, k=4)
    shown = ", ".join(f"{w} ({f})" for w, f in ranked) or "(no completions)"
    print(f"typing {prefix!r:>7} -> {shown}")

# Why $O(L)$ is the headline: search cost is independent of how many keys exist.
big = Trie()
for i in range(20000):
    big.insert(f"key{i:05d}")
print(f"\n{len(big)} keys, {big.node_count()} nodes")
print("steps to search 'key19999':", len("key19999"),
      "- the same as in the 15-word trie")
print("prefix 'key199' has", len(big.collect("key199")), "completions")

Compare the `hel` row with the alphabetical listing above it. Alphabetically the first completion of `hel` is `held`; by frequency it is `hello`, and `held` drops to third. That one sort is the difference between a search box people use and one they fight — and note that ties break alphabetically, so the ordering is still deterministic.

The honest cost of a trie is **space**. Our fifteen short words needed twenty-four nodes, each a Python object with a dictionary — vastly more memory than fifteen strings in a set. Two standard remedies: a **radix trie** (or Patricia trie) collapses every chain of single-child nodes into one edge holding a whole substring, which is a large win on sparse key sets like URLs; and for a fixed alphabet you can replace the per-node dictionary with an array. Reach for a trie when you need prefix queries, ordered traversal, or bounded-by-key-length worst cases — not when a `set` would do.

### Micro-exercise: longest common prefix

Write `longest_common_prefix(trie)` that walks down from the root while there is exactly one child and no word ends at the current node, and returns the string it accumulated. On a trie of `["interspecies", "interstellar", "interstate"]` it should return `"inters"`.

In [ ]:
def longest_common_prefix(t):
    node, out = t.root, ""
    # your code here: while len(node.children) == 1 and not node.is_word,
    # take that one child, append its character, and descend
    return out

# Uncomment to test:
# t = Trie()
# for w in ["interspecies", "interstellar", "interstate"]:
#     t.insert(w)
# assert longest_common_prefix(t) == "inters"
# print("longest common prefix:", longest_common_prefix(t))

## 8. Skip lists: balance by coin flip

A sorted linked list has a perfect ordering and a terrible search: you must walk. A **skip list** adds express lanes. Every node lives at level 0; some also appear at level 1, fewer at level 2, and so on. Searching starts at the top-left, runs right until the next key would overshoot, then drops a level and repeats — halving the remaining distance at each level, exactly like binary search.

The remarkable part is how the levels are chosen: **by flipping a coin**. A new node gets level 1, then keeps promoting itself while the coin comes up heads. No rotations, no colours, no rebalancing — the *distribution* does the balancing, giving expected $O(\log n)$ search with no worst-case guarantee at all. That trade (expected instead of guaranteed, in exchange for code you can hold in your head) is why skip lists appear in Redis sorted sets and in several storage engines' memtables.

In [ ]:
class SkipNode:
    __slots__ = ("key", "value", "forward")
    def __init__(self, key, value, level):
        self.key, self.value = key, value
        self.forward = [None] * (level + 1)      # one pointer per level

class SkipList:
    def __init__(self, seed=36, p=0.5, max_level=16):
        self.p, self.max_level = p, max_level
        self.head = SkipNode(None, None, max_level)     # sentinel, all levels
        self.level = 0                                  # highest level in use
        self.rng = random.Random(seed)
        self.n = 0

    def random_level(self):
        """Level 0, then promote while the coin says heads."""
        lvl = 0
        while self.rng.random() < self.p and lvl < self.max_level:
            lvl += 1
        return lvl

    def _predecessors(self, key):
        """For each level, the last node whose key is < `key`."""
        update = [self.head] * (self.max_level + 1)
        node = self.head
        for lvl in range(self.level, -1, -1):
            while (node.forward[lvl] is not None
                   and node.forward[lvl].key < key):
                node = node.forward[lvl]
            update[lvl] = node
        return update, node.forward[0]

    def search(self, key, default=None, count_steps=False):
        steps, node = 0, self.head
        for lvl in range(self.level, -1, -1):
            while node.forward[lvl] is not None and node.forward[lvl].key < key:
                node = node.forward[lvl]
                steps += 1
            steps += 1                       # the comparison that stopped us
        found = node.forward[0]
        value = found.value if found is not None and found.key == key else default
        return (value, steps) if count_steps else value

    def insert(self, key, value):
        update, nxt = self._predecessors(key)
        if nxt is not None and nxt.key == key:
            nxt.value = value
            return 0
        lvl = self.random_level()
        if lvl > self.level:                 # the list grew a new top lane
            self.level = lvl
        node = SkipNode(key, value, lvl)
        for i in range(lvl + 1):
            node.forward[i] = update[i].forward[i]
            update[i].forward[i] = node
        self.n += 1
        return lvl

    def keys(self):
        out, node = [], self.head.forward[0]
        while node is not None:
            out.append(node.key)
            node = node.forward[0]
        return out

    def height_of(self, key):
        _, nxt = self._predecessors(key)
        return len(nxt.forward) - 1 if nxt is not None and nxt.key == key else None

    def render(self):
        """One line per level, highest first - the express lanes made visible."""
        base = self.keys()
        for lvl in range(self.level, -1, -1):
            present, node = set(), self.head.forward[lvl]
            while node is not None:
                present.add(node.key)
                node = node.forward[lvl]
            row = "".join(f"{k:>4}" if k in present else "   ." for k in base)
            print(f"  L{lvl}: {row}")

sl = SkipList(seed=1)
for k in [3, 6, 7, 9, 12, 17, 19, 21, 25, 26]:
    sl.insert(k, f"v{k}")
print(f"{sl.n} keys, top level in use: {sl.level}")
sl.render()
print("\nkeys in order:", sl.keys())
print("search(19) ->", sl.search(19), " search(20) ->", sl.search(20))
print("levels:", {k: sl.height_of(k) for k in sl.keys()})

Read the rendering bottom-up. Level 0 holds everything, and each level above holds a thinning subset. A search for a key near the right-hand end starts on the top lane and skips most of the list in a couple of hops before dropping down to walk the last few nodes.

Now the distribution itself, which is where the $O(\log n)$ comes from. With $p = 1/2$, the probability of reaching level $\ell$ is $2^{-\ell}$, so half the nodes are level 0, a quarter level 1, an eighth level 2 — the expected number of pointers per node is exactly 2, and the expected top level is $\log_2 n$.

In [ ]:
probe = SkipList(seed=7)
levels = [probe.random_level() for _ in range(20000)]
observed = [levels.count(l) / len(levels) for l in range(9)]
expected = [0.5 ** (l + 1) for l in range(9)]

print(f"{'level':>6}{'observed':>11}{'predicted 2^-(l+1)':>21}")
for l in range(9):
    print(f"{l:>6}{observed[l]:>11.4f}{expected[l]:>21.4f}")
print(f"\nmean pointers per node: {1 + sum(levels) / len(levels):.3f} "
      f"(theory: {1 / (1 - 0.5):.3f})")

fig, ax = plt.subplots(figsize=(7.0, 3.4))
xs = list(range(9))
ax.bar([x - 0.2 for x in xs], observed, width=0.4, label="observed (20000 draws)")
ax.bar([x + 0.2 for x in xs], expected, width=0.4, label="predicted $2^{-(\\ell+1)}$")
ax.set_xlabel("node level")
ax.set_ylabel("fraction of nodes")
ax.set_title("Skip-list levels come from a geometric distribution")
ax.set_xticks(xs)
ax.legend()
fig.tight_layout()

# And the payoff: search cost against list length, averaged over 100 lookups.
print(f"\n{'n':>7}{'top level':>11}{'log2(n)':>10}{'mean steps per search':>23}")
for n in (100, 1000, 10000):
    s = SkipList(seed=1)
    for k in range(n):
        s.insert(k, k)
    probes = random.Random(0).sample(range(n), 100)
    mean_steps = sum(s.search(k, count_steps=True)[1] for k in probes) / 100
    print(f"{n:>7}{s.level:>11}{math.log2(n):>10.1f}{mean_steps:>23.1f}")

The observed frequencies track $2^{-(\ell+1)}$ closely, and the mean pointer count lands on 2 — so a skip list costs about twice the memory of a plain linked list and buys a logarithmic search with it. The last table is the payoff: multiply $n$ by ten and the mean search cost grows by about five steps, not by a factor of ten.

The `top level` column is worth a second look, because it is higher than $\log_2 n$. `max_level` is only a cap; one unlucky node that flipped thirteen heads in a row creates express lanes holding a single key each, and every search pays a descent through each of them. It is a constant few steps, not a complexity problem — but it is why real implementations cap `max_level` near $\log_2$ of the largest size they expect (Redis uses 32) rather than leaving it enormous.

The honest caveat is in the word *expected*. There is no rotation enforcing anything here, so an unlucky sequence of coin flips can produce a skip list that is essentially a linked list. The probability is vanishingly small and it is not adversarially exploitable if the randomness is not attacker-controlled — but if you need a hard guarantee, you need a balanced tree.

## What you built

| Structure | Invariant | Cost | Reach for it when |
|---|---|---|---|
| plain BST | left < node < right | $O(h)$, and $h$ can be $n$ | never, for untrusted input order |
| rotation | in-order sequence preserved | $O(1)$ | you need to reshape a BST |
| AVL | $\lvert h_L - h_R \rvert \le 1$ | $O(\log n)$, height $\le 1.44\log_2 n$ | lookups dominate |
| red-black | five colour properties | $O(\log n)$, $\le 2$ rotations/insert | updates dominate |
| hash table | good spread, load factor bounded | $O(1)$ expected | exact-key lookup, no ordering needed |
| open addressing | tombstones, not holes | $O(1)$ expected | cache locality matters |
| trie | key is the path, `is_word` marks ends | $O(L)$, independent of $n$ | prefix queries, autocomplete |
| skip list | level distribution is geometric | $O(\log n)$ expected | you want ordered ops without rebalancing code |

## Try it yourself

Bigger exercises. Every scaffold runs as-is.

### Exercise 1 — Which arrival orders actually hurt?

Sorted input is the worst case, but it is not the only bad one. Measure the plain-BST height for several arrival orders of the same 1023 keys: sorted, reverse sorted, shuffled, "sorted in two halves" (`1..512` then `1023..513`), and "sawtooth" (alternating smallest and largest remaining). Print a table, and for each order also build an `AVLTree` and print its height beside it. Which orders are bad for the plain BST, and does *any* order hurt the AVL tree?

In [ ]:
def sawtooth(n):
    lo, hi, out = 1, n, []
    while lo <= hi:
        out.append(lo); lo += 1
        if lo <= hi:
            out.append(hi); hi -= 1
    return out

ORDERS = {
    "sorted": list(range(1, 1024)),
    "reverse": list(range(1023, 0, -1)),
    "shuffled": random.Random(7).sample(range(1, 1024), 1023),
    "two halves": list(range(1, 513)) + list(range(1023, 512, -1)),
    "sawtooth": sawtooth(1023),
}

# your code here: for each order, print the plain-BST height and the AVL height
for label, order in ORDERS.items():
    print(f"{label:<12} plain BST height ?, AVL height ?")

### Exercise 2 — Load factor is the only number that matters

For a chained hash table, the expected length of the chain you land in is the **load factor** $\alpha = n/m$, and the expected comparisons for a successful search is about $1 + \alpha/2$. Verify it: build chained tables with `djb2` at load factors 0.25, 0.5, 1.0, 2.0 and 4.0, measure the mean number of comparisons for a successful lookup over all keys, and compare with the formula. Then plot measured against predicted, labelling both axes.

The lesson to extract: the table's *absolute size* never appears in that formula. A table with a million entries and $\alpha = 0.75$ is exactly as fast as one with a hundred.

In [ ]:
def chain_experiment(alpha, words=WORDS, seed=0):
    """Build a chained table at the given load factor; return mean probe count."""
    m = max(1, int(len(words) / alpha))
    buckets = [[] for _ in range(m)]
    for w in words:
        buckets[djb2(w) % m].append(w)
    # your code here: for each word, its probe count is 1 + its index in its
    # bucket; return the mean over all words
    return 0.0

# Uncomment to test:
# print(f"{'alpha':>6}{'measured':>11}{'1 + alpha/2':>14}")
# for alpha in (0.25, 0.5, 1.0, 2.0, 4.0):
#     print(f"{alpha:>6.2f}{chain_experiment(alpha):>11.2f}{1 + alpha / 2:>14.2f}")

### Exercise 3 — AVL deletion, and the rotation it costs

Add `delete(key)` to `AVLTree`. The three cases are Chapter 20's: a leaf just goes; a node with one child is replaced by that child; a node with two children is replaced by its **in-order successor** (the leftmost node of its right subtree), and then you delete the successor from the right subtree. The AVL part is one line — `return self._rebalance(node)` on the way back up.

Then measure the claim that deletion can cost more rotations than insertion: build a tree from `1..255`, record `tree.rotations`, delete the keys in a random order, and report the rotations that deletion cost. Assert `is_avl` after **every** deletion — that assertion is what will find your bug.

In [ ]:
def avl_min_node(node):
    while node.left is not None:
        node = node.left
    return node

def avl_delete(tree, node, key):
    if node is None:
        return None
    if key < node.key:
        node.left = avl_delete(tree, node.left, key)
    elif key > node.key:
        node.right = avl_delete(tree, node.right, key)
    else:
        # your code here: the three cases, then fall through to _rebalance
        pass
    return tree._rebalance(node) if node is not None else None

# Uncomment to test once the three cases are written:
# t = AVLTree()
# for k in range(1, 256):
#     t.insert(k)
# after_build = t.rotations
# for k in random.Random(3).sample(range(1, 256), 100):
#     t.root = avl_delete(t, t.root, k)
#     assert is_avl(t.root), f"invariant broken after deleting {k}"
# print("rotations: build", after_build, " delete", t.rotations - after_build)

### Exercise 4 — Autocomplete that does not scan the whole subtree

`suggest` collects *every* completion and then sorts. For the prefix `"a"` on a real dictionary that is hundreds of thousands of candidates for five suggestions — quadratic-feeling work for a linear-feeling answer.

Fix it two ways and compare. **(a)** Keep a heap of size $k$ during the traversal instead of a list of everything (Chapter 21's top-$k$ trick), so memory is $O(k)$. **(b)** Store, at every node, the maximum frequency anywhere in its subtree; then a best-first search using a priority queue over nodes can stop as soon as $k$ words have been emitted, without visiting the rest of the subtree at all. Count nodes visited under each scheme for the prefix `"key1"` on the 20,000-key `big` trie and print the ratio.

In [ ]:
import heapq

def suggest_heap(trie, prefix, k=5):
    """(a) O(k) memory: keep only the best k seen so far."""
    node = trie._node_for(prefix)
    best, visited = [], 0
    if node is None:
        return [], 0
    # your code here: DFS from `node`, counting `visited`, and use
    # heapq.heappush / heappushpop on `best` to keep only k items
    return sorted(best, key=lambda wf: (-wf[1], wf[0])), visited

# Uncomment to test:
# print(suggest_heap(trie, "he", k=3))
# print(suggest_heap(big, "key1", k=5))